In [ ]:
# -------------------------------
# 1. Load processed EDA dataset
# -------------------------------

import pandas as pd
import numpy as np

df_eda = pd.read_csv("../data/processed/df_eda.csv")
print("Loaded dataset shape:", df_eda.shape)
display(df_eda.head())

In [ ]:
# Define features and targets
# proto_features = [col for col in df_eda.columns if col.startswith("proto_")]
features = ["log_byte_ratio", "log_dur", "log_sbytes", "log_dbytes", "log_sbytes_per_dur", "sttl", "ttl_diff", "dttl"] # + proto_features
X = df_eda[features]
y = df_eda["Label"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

In [ ]:
# -------------------------------
# 2. Fit isolation forest
# -------------------------------

from sklearn.ensemble import IsolationForest

contamination = df_eda["Label"].sum()/len(df_eda)
print("Contamination fraction:", contamination)
# Model initialisation
model = IsolationForest(
    n_estimators=600,
    max_samples=256,
    contamination=0.038316,
    random_state=42
)

# Model fit
model.fit(X)

In [ ]:
# Hyperparameter search
from sklearn.metrics import classification_report
import itertools

n_estimators_list = [100, 300, 500]
max_samples_list = [256, 512, 1024]
contamination_list = [y.sum()/len(y)*0.8, y.sum()/len(y), y.sum()/len(y)*1.2]

results = []

for n, ms, cont in itertools.product(n_estimators_list, max_samples_list, contamination_list):
    model = IsolationForest(
        n_estimators=n,
        max_samples=ms,
        contamination=cont,
        random_state=42
    )
    model.fit(X)
    
    y_pred = model.predict(X)
    y_pred = pd.Series(y_pred).map({1: 0, -1: 1})
    
    report = classification_report(y, y_pred, output_dict=True)
    
    results.append({
        "n_estimators": n,
        "max_samples": ms,
        "contamination": cont,
        "precision_attack": report["1"]["precision"],
        "recall_attack": report["1"]["recall"],
        "f1_attack": report["1"]["f1-score"]
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values(by="f1_attack", ascending=False).head(10))

In [ ]:
# -------------------------------
# 3. Generate predictions and anomaly scores
# -------------------------------

anomaly_scores = model.score_samples(X)

pred_labels = model.predict(X)
pred_labels_binary = (pred_labels == -1).astype(int)

df_eda["anomaly_score"] = anomaly_scores
df_eda["pred_anomaly"] = pred_labels_binary

df_eda[["Label", "anomaly_score", "pred_anomaly"]].head()

In [ ]:
# -------------------------------
# 4. Evaluate and visualise predictions
# -------------------------------

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Quick stats
num_anomalies = df_eda["pred_anomaly"].sum()
print(f"Number of predicted anomalies: {num_anomalies}")

# Confusion matrix
cm = confusion_matrix(df_eda["Label"], df_eda["pred_anomaly"])
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix: Isolation Forest vs True Labels")
plt.show()

# Classification report
print(classification_report(df_eda["Label"], df_eda["pred_anomaly"]))

In [ ]:
# Visualise anomalies in feature space
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_eda,
    x="log_byte_ratio",
    y="log_dur",
    hue="Label",
    style="pred_anomaly",
    alpha=0.6,
    palette={0: "blue", 1: "red"}
)
plt.xlabel("Log(Byte Ratio)")
plt.ylabel("Log(Duration)")
plt.title("Predicted Anomalies vs True Attack Labels")
plt.show()

In [ ]:
# Create outcome categories
df_eda["outcome"] = "TN" # default
df_eda.loc[(df_eda["Label"] == 1) & (df_eda["pred_anomaly"] == 1), "outcome"] = "TP"
df_eda.loc[(df_eda["Label"] == 1) & (df_eda["pred_anomaly"] == 0), "outcome"] = "FN"
df_eda.loc[(df_eda["Label"] == 0) & (df_eda["pred_anomaly"] == 1), "outcome"] = "FP"
# Sanity check
display(df_eda["outcome"].value_counts())

In [ ]:
attack_subset = df_eda[df_eda["Label"] == 1]
display(
    attack_subset.groupby("outcome")[
        [
            "log_byte_ratio",
            "log_dur",
            "log_sbytes",
            "log_dbytes",
            "log_sbytes_per_dur",
            "log_dbytes_per_dur"
        ]
    ].median()
)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=attack_subset,
    x="log_byte_ratio",
    y="log_dur",
    hue="outcome",
    palette={"TP": "red", "FN": "orange"},
    alpha=0.6
)
plt.xlabel("Log(Byte Ratio)")
plt.ylabel("Log(Duration)")
plt.title("Detected vs Missed Attacks (TP vs FN)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=attack_subset,
    x="log_sbytes_per_dur",
    y="log_dbytes_per_dur",
    hue="outcome",
    palette={"TP": "red", "FN": "orange"},
    alpha=0.6
)
plt.xlabel("Log(Source Bytes per Duration)")
plt.ylabel("Log(Destination Bytes per Duration)")
plt.title("Detected vs Missed Attacks (TP vs FN) by Flow Rates")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
attack_subset["avg_d_pkt_size"] = attack_subset["dbytes"]/attack_subset["Dpkts"].replace(0, np.nan)
attack_subset["log_avg_d_pkt_size"] = np.log1p(attack_subset["avg_d_pkt_size"])
attack_subset["log_avg_d_pkt_size"] = attack_subset["log_avg_d_pkt_size"].fillna(0)
sns.boxplot(
    x="outcome",
    y="log_avg_d_pkt_size",
    data=attack_subset
)
plt.title("Distribution of log_avg_d_pkt_size for TPs vs FNs")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(
    x="outcome",
    y="d_to_s_pkt_ratio",
    data=attack_subset
)
plt.title("Distribution of d_to_s_pkt_ratio for TPs vs FNs")
plt.show()

In [ ]:
# Distinguish FNs from normal flows
fn_subset = df_eda[df_eda["outcome"] == "FN"]
normal_subset = df_eda[df_eda["Label"] == 0]

# Pick numeric features
numeric_features = [col for col in df_eda.columns if df_eda[col].dtype in [float, int]]

# Safe numeric subsets
fn_safe = fn_subset[numeric_features].replace([np.inf, -np.inf], np.nan).fillna(0)
normal_safe = normal_subset[numeric_features].replace([np.inf, -np.inf], np.nan).fillna(0)

def cohens_d(x, y):
    """Compute Cohen's d for two arrays."""
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((x.std(ddof=1) ** 2) * (nx - 1) + (y.std(ddof=1) ** 2) * (ny - 1)) / dof)
    if pooled_std == 0:
        return 0
    return (x.mean() - y.mean()) / pooled_std

effect_sizes = {}
for feat in numeric_features:
    effect_sizes[feat] = cohens_d(fn_safe[feat], normal_safe[feat])

effect_sizes_df = pd.DataFrame.from_dict(effect_sizes, orient='index', columns=['cohens_d'])
effect_sizes_df['abs_d'] = effect_sizes_df['cohens_d'].abs()
effect_sizes_df = effect_sizes_df.sort_values(by='abs_d', ascending=False)

print(effect_sizes_df.head(10))

In [ ]:
# Attacks vs normal flows
attack_subset = df_eda[df_eda["Label"] == 1]
normal_subset = df_eda[df_eda["Label"] == 0]

# Compute Cohen's d as before
effect_sizes_all = {}
for feat in numeric_features:
    effect_sizes_all[feat] = cohens_d(
        attack_subset[feat].replace([np.inf, -np.inf], np.nan).fillna(0),
        normal_subset[feat].replace([np.inf, -np.inf], np.nan).fillna(0)
    )

effect_sizes_all_df = pd.DataFrame.from_dict(effect_sizes_all, orient='index', columns=['cohens_d'])
effect_sizes_all_df['abs_d'] = effect_sizes_all_df['cohens_d'].abs()
effect_sizes_all_df = effect_sizes_all_df.sort_values(by='abs_d', ascending=False)
print(effect_sizes_all_df.head(10))